# STEA — Colab Notebook for Temporal Cross-Lingual Entity Alignment

This notebook implements the **STEA (Simple Temporal Information Matching Mechanism for Entity Alignment between Temporal Knowledge Graphs)** method of Cai et al. (COLING 2022) in PyTorch, adapted for the user's temporal cross-lingual `fr_en` dataset.

**Important:** The paper's official code is available at `lcai2/STEA`. This notebook is a self-contained PyTorch implementation designed to run in Google Colab and to use the same core STEA ideas:
- structural + relational feature fusion
- simple GNN aggregation
- global-aware multi-layer entity representation
- Manhattan-distance triplet loss
- temporal time dictionaries
- temporal similarity + embedding similarity
- CSLS prediction
- optional iterative mutual-nearest-neighbor seed expansion

The paper reports `d=100`, GNN depth `L=2`, margin `λ=3`, dropout `0.3`, `α=0.3`, 5 iterations, and 1200 epochs/iteration as its default settings.

Source paper: Cai et al., COLING 2022.


In [1]:
# ============================================================
# 1. Imports, reproducibility, and GPU check
# ============================================================

import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. In Colab use Runtime -> Change runtime type -> GPU.")


PyTorch: 2.11.0+cpu
Device: cpu


In [2]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 3. Configuration
# ============================================================

# Your existing Bai temporal cross-lingual dataset
RAW_SRC_DIR = Path("/content/drive/MyDrive/dbpt15k/zh_en")

# Local Colab copy
DATA_DIR = Path("/content/stea_data/zh_en")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- STEA settings from the paper ----------------
TRAIN_SEED = 200       # number of supervised seed pairs used for training
DIM = 100
DEPTH = 2
MARGIN = 3.0
DROPOUT = 0.3
ALPHA = 0.3            # temporal similarity weight
LR = 0.005
EPOCHS_PER_ITER = 1200
ITERATIONS = 5
BATCH_SIZE = 1024

# Evaluation
CSLS_K = 10
TOP_K = (1, 5, 10)

# Run the full iterative STEA procedure.
# Set False for STEA* (non-iterative version).
USE_ITERATIVE_STEA = True

print("RAW_SRC_DIR:", RAW_SRC_DIR)
print("DATA_DIR:", DATA_DIR)
print("DEVICE:", DEVICE)


RAW_SRC_DIR: /content/drive/MyDrive/dbpt15k/zh_en
DATA_DIR: /content/stea_data/zh_en
DEVICE: cpu


In [4]:
# ============================================================
# 4. Copy the four dataset files
# ============================================================

required = ["triples_1", "triples_2", "train", "test"]

for fname in required:
    src = RAW_SRC_DIR / fname
    if not src.exists():
        raise FileNotFoundError(f"Missing file: {src}")
    shutil.copy2(src, DATA_DIR / fname)

# Use names matching the TEA-GNN/STEA convention
shutil.copy2(DATA_DIR / "train", DATA_DIR / "sup_pairs")
shutil.copy2(DATA_DIR / "test", DATA_DIR / "ref_pairs")

print("Copied:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")


Copied:
  ref_pairs 118208 bytes
  sup_pairs 45599 bytes
  test 118208 bytes
  train 45599 bytes
  triples_1 1590595 bytes
  triples_2 2365712 bytes


In [5]:
# ============================================================
# 5. Dataset inspection
# ============================================================

def read_temporal_triples(path):
    triples = []
    entities = set()
    relations = set()
    times = set()

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            parts = line.split()
            if not parts:
                continue

            if len(parts) == 4:
                h, r, t, tau = map(int, parts)
                ts, te = tau, tau
            elif len(parts) == 5:
                h, r, t, ts, te = map(int, parts)
            else:
                raise ValueError(
                    f"{path.name}: line {line_no} has {len(parts)} fields; "
                    "expected 4 or 5."
                )

            triples.append((h, r, t, ts, te))
            entities.update((h, t))
            relations.add(r)

            # STEA constructs the time dictionary from timestamps.
            # For an interval, the paper adds the start and end timestamps.
            times.update((ts, te))

    return triples, entities, relations, times


triples1, e1_raw, r1_raw, t1_raw = read_temporal_triples(DATA_DIR / "triples_1")
triples2, e2_raw, r2_raw, t2_raw = read_temporal_triples(DATA_DIR / "triples_2")

print("KG1")
print("  triples:", len(triples1))
print("  entities:", len(e1_raw))
print("  relations:", len(r1_raw))
print("  times:", len(t1_raw))
print("  entity ID range:", (min(e1_raw), max(e1_raw)))
print("  relation ID range:", (min(r1_raw), max(r1_raw)))

print("\nKG2")
print("  triples:", len(triples2))
print("  entities:", len(e2_raw))
print("  relations:", len(r2_raw))
print("  times:", len(t2_raw))
print("  entity ID range:", (min(e2_raw), max(e2_raw)))
print("  relation ID range:", (min(r2_raw), max(r2_raw)))

print("\nRAW entity-ID overlap:", len(e1_raw & e2_raw))
print("RAW relation-ID overlap:", len(r1_raw & r2_raw))


KG1
  triples: 70414
  entities: 19388
  relations: 1701
  times: 172
  entity ID range: (0, 29887)
  relation ID range: (0, 1700)

KG2
  triples: 95142
  entities: 19572
  relations: 1323
  times: 185
  entity ID range: (10500, 38959)
  relation ID range: (1701, 3023)

RAW entity-ID overlap: 0
RAW relation-ID overlap: 0


In [6]:
# ============================================================
# 6. Load alignment pairs and verify namespaces
# ============================================================

def load_pairs(path):
    arr = np.loadtxt(path, dtype=np.int64)
    arr = np.atleast_2d(arr)
    if arr.shape[1] != 2:
        raise ValueError(f"{path} should contain two integer columns.")
    return arr


train_all = load_pairs(DATA_DIR / "sup_pairs")
test_pairs = load_pairs(DATA_DIR / "ref_pairs")

print("Original train pairs:", train_all.shape)
print("Original test pairs :", test_pairs.shape)

print("\nTrain left IDs missing from KG1:",
      sum(x not in e1_raw for x in train_all[:, 0]))
print("Train right IDs missing from KG2:",
      sum(x not in e2_raw for x in train_all[:, 1]))
print("Test left IDs missing from KG1:",
      sum(x not in e1_raw for x in test_pairs[:, 0]))
print("Test right IDs missing from KG2:",
      sum(x not in e2_raw for x in test_pairs[:, 1]))

# Same protocol used in the previous TEA-GNN/TREA experiments:
# first TRAIN_SEED pairs = supervised seeds;
# remaining original train pairs + original test pairs = evaluation pool.
if len(train_all) < TRAIN_SEED:
    raise ValueError(
        f"TRAIN_SEED={TRAIN_SEED}, but only {len(train_all)} training pairs exist."
    )

seed_pairs_raw = train_all[:TRAIN_SEED]
eval_pairs_raw = np.vstack([train_all[TRAIN_SEED:], test_pairs])

print("\nSTEA seed pairs:", len(seed_pairs_raw))
print("STEA evaluation pairs:", len(eval_pairs_raw))


Original train pairs: (4050, 2)
Original test pairs : (10500, 2)

Train left IDs missing from KG1: 0
Train right IDs missing from KG2: 0
Test left IDs missing from KG1: 0
Test right IDs missing from KG2: 0

STEA seed pairs: 200
STEA evaluation pairs: 14350


In [7]:
# ============================================================
# 7. Remap entity/relation IDs safely
# ============================================================
# IMPORTANT:
# The two KGs can have local/overlapping integer IDs. We therefore create
# separate namespaces and then offset KG2. This avoids accidentally
# collapsing two different entities just because they share an integer ID.
#
# Relations are also kept in separate namespaces because STEA learns
# relation embeddings independently for each KG.

def make_entity_maps(entity_set):
    raw_sorted = sorted(entity_set)
    raw_to_local = {x: i for i, x in enumerate(raw_sorted)}
    local_to_raw = {i: x for i, x in enumerate(raw_sorted)}
    return raw_to_local, local_to_raw


e1_to_local, local_to_e1 = make_entity_maps(e1_raw)
e2_to_local, local_to_e2 = make_entity_maps(e2_raw)

r1_to_local, local_to_r1 = make_entity_maps(r1_raw)
r2_to_local, local_to_r2 = make_entity_maps(r2_raw)

N1 = len(e1_raw)
N2 = len(e2_raw)
R1 = len(r1_raw)
R2 = len(r2_raw)

def remap_triples(triples, e_map, r_map, entity_offset=0, relation_offset=0):
    out = []
    for h, r, t, ts, te in triples:
        out.append((
            e_map[h] + entity_offset,
            r_map[r] + relation_offset,
            e_map[t] + entity_offset,
            ts,
            te
        ))
    return out


triples1_m = remap_triples(triples1, e1_to_local, r1_to_local, 0, 0)
triples2_m = remap_triples(triples2, e2_to_local, r2_to_local, N1, R1)

seed_pairs = np.array([
    [e1_to_local[a], e2_to_local[b] + N1]
    for a, b in seed_pairs_raw
], dtype=np.int64)

eval_pairs = np.array([
    [e1_to_local[a], e2_to_local[b] + N1]
    for a, b in eval_pairs_raw
], dtype=np.int64)

NUM_ENTITIES = N1 + N2
NUM_RELATIONS = R1 + R2

print("KG1 entities:", N1)
print("KG2 entities:", N2)
print("Total entity nodes:", NUM_ENTITIES)
print("KG1 relations:", R1)
print("KG2 relations:", R2)
print("Total relation embeddings:", NUM_RELATIONS)
print("Seed pairs:", seed_pairs.shape)
print("Evaluation pairs:", eval_pairs.shape)


KG1 entities: 19388
KG2 entities: 19572
Total entity nodes: 38960
KG1 relations: 1701
KG2 relations: 1323
Total relation embeddings: 3024
Seed pairs: (200, 2)
Evaluation pairs: (14350, 2)


In [8]:
# ============================================================
# 8. Build STEA graph and time dictionaries
# ============================================================

# The paper uses an adjacency matrix with self-loops and D^{-1} A H.
# We keep the two KGs disconnected during message passing.

neighbors = [set() for _ in range(NUM_ENTITIES)]
entity_relations = [set() for _ in range(NUM_ENTITIES)]
time_dict = [set() for _ in range(NUM_ENTITIES)]

for h, r, t, ts, te in triples1_m + triples2_m:
    # Structural neighborhood
    neighbors[h].add(t)
    neighbors[t].add(h)

    # Relations incident to the entity
    entity_relations[h].add(r)
    entity_relations[t].add(r)

    # STEA time dictionary:
    # point -> one timestamp; interval -> start and end.
    time_dict[h].add(ts)
    time_dict[h].add(te)
    time_dict[t].add(ts)
    time_dict[t].add(te)

# Add self loops for graph convolution.
# Use a sparse COO matrix for D^{-1}A.
rows = []
cols = []

for i in range(NUM_ENTITIES):
    rows.append(i)
    cols.append(i)
    for j in neighbors[i]:
        rows.append(i)
        cols.append(j)

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)

degree = np.bincount(rows, minlength=NUM_ENTITIES).astype(np.float32)
values = 1.0 / degree[rows]

indices = torch.tensor(np.vstack([rows, cols]), dtype=torch.long)
values_t = torch.tensor(values, dtype=torch.float32)

adj = torch.sparse_coo_tensor(
    indices,
    values_t,
    size=(NUM_ENTITIES, NUM_ENTITIES)
).coalesce().to(DEVICE)

print("Sparse adjacency shape:", adj.shape)
print("Sparse adjacency nnz:", adj._nnz())
print("Average degree incl. self-loop:", degree.mean())

print("\nExample time dictionaries:")
for idx in list(range(min(3, NUM_ENTITIES))):
    print(idx, sorted(time_dict[idx])[:20])


Sparse adjacency shape: torch.Size([38960, 38960])
Sparse adjacency nnz: 298096
Average degree incl. self-loop: 7.661833

Example time dictionaries:
0 [0, 188, 189, 197, 198, 199, 211, 215, 218, 219, 220, 225, 228, 229, 232]
1 [195, 197, 214, 215, 221, 222, 223, 224, 225, 226, 228, 231]
2 [0, 215, 225, 226, 230]


/tmp/ipykernel_57072/4197144060.py:49: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(


In [9]:
# ============================================================
# 9. STEA model
# ============================================================

class STEA(nn.Module):
    def __init__(self, num_entities, num_relations, dim=100, depth=2, dropout=0.3):
        super().__init__()

        self.num_entities = num_entities
        self.num_relations = num_relations
        self.dim = dim
        self.depth = depth

        self.entity_emb = nn.Parameter(
            torch.empty(num_entities, dim)
        )
        self.relation_emb = nn.Parameter(
            torch.empty(num_relations, dim)
        )

        nn.init.xavier_uniform_(self.entity_emb)
        nn.init.xavier_uniform_(self.relation_emb)

        self.dropout = nn.Dropout(dropout)

    def forward(self, adj):
        # Equation (1): average neighboring entity embeddings.
        # Equation (2): average incident relation embeddings.
        structural = torch.sparse.mm(
            adj, self.entity_emb
        )

        # Relation feature aggregation is calculated explicitly because
        # relation features are associated with incident KG edges.
        rel_sum = torch.zeros_like(self.entity_emb)
        rel_count = torch.zeros(
            self.num_entities, 1,
            device=self.entity_emb.device
        )

        # These are populated externally by the precomputed relation
        # incidence matrix.
        relational = torch.sparse.mm(
            relation_incidence.to(self.entity_emb.device),
            self.relation_emb
        )

        # Average over incident relations.
        relational = relational / relation_count.to(self.entity_emb.device).clamp_min(1.0)

        # Eq. (3): feature fusion.
        h = torch.cat([structural, relational], dim=1)

        # Eq. (5): global-aware representation.
        representations = [h]

        for _ in range(self.depth - 1):
            h = torch.sparse.mm(adj, h)
            h = F.relu(h)
            h = self.dropout(h)
            representations.append(h)

        return torch.cat(representations, dim=1)


# Build sparse entity-relation incidence matrix.
er_rows = []
er_cols = []
for entity_id, rels in enumerate(entity_relations):
    for rel_id in rels:
        er_rows.append(entity_id)
        er_cols.append(rel_id)

if er_rows:
    er_values = np.ones(len(er_rows), dtype=np.float32)
    er_indices = torch.tensor(
        np.vstack([er_rows, er_cols]), dtype=torch.long
    )
    relation_incidence = torch.sparse_coo_tensor(
        er_indices,
        torch.tensor(er_values),
        size=(NUM_ENTITIES, NUM_RELATIONS)
    ).coalesce().to(DEVICE)

relation_count = torch.zeros(NUM_ENTITIES, 1, device=DEVICE)
for entity_id, rels in enumerate(entity_relations):
    relation_count[entity_id, 0] = max(len(rels), 1)

model = STEA(
    NUM_ENTITIES,
    NUM_RELATIONS,
    dim=DIM,
    depth=DEPTH,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.RMSprop(
    model.parameters(),
    lr=LR
)

print(model)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


STEA(
  (dropout): Dropout(p=0.3, inplace=False)
)
Trainable parameters: 4198400


In [10]:
# ============================================================
# 10. STEA triplet loss
# ============================================================

def stea_triplet_loss(emb, pairs, margin=3.0):
    """
    Paper Eq. (6):
        max(dist(pos) - dist(neg) + margin, 0)

    Distance is Manhattan/L1 distance, Eq. (7).

    A negative pair is created by randomly replacing either the source
    or target entity.
    """

    if len(pairs) == 0:
        return emb.sum() * 0.0

    device = emb.device
    p = torch.as_tensor(pairs, dtype=torch.long, device=device)

    src = p[:, 0]
    tgt = p[:, 1]

    # Randomly corrupt one side.
    corrupt_source = torch.rand(len(p), device=device) < 0.5

    neg_src = src.clone()
    neg_tgt = tgt.clone()

    rand_src = torch.randint(
        0, N1, (len(p),), device=device
    )
    rand_tgt = torch.randint(
        N1, N1 + N2, (len(p),), device=device
    )

    neg_src[corrupt_source] = rand_src[corrupt_source]
    neg_tgt[~corrupt_source] = rand_tgt[~corrupt_source]

    pos_dist = torch.abs(emb[src] - emb[tgt]).sum(dim=1)
    neg_dist = torch.abs(emb[neg_src] - emb[neg_tgt]).sum(dim=1)

    loss = F.relu(pos_dist - neg_dist + margin)
    return loss.mean()


In [11]:
# ============================================================
# 11. Temporal similarity
# ============================================================

def temporal_similarity_matrix(source_ids, target_ids, time_dicts):
    """
    STEA Eq. (8):
        st(i,j) = 2*c / (m+n)

    where c is the number of matching time-dictionary items.
    """
    S = np.zeros((len(source_ids), len(target_ids)), dtype=np.float32)

    target_sets = [time_dict[int(j)] for j in target_ids]

    for i, src in enumerate(source_ids):
        s = time_dict[int(src)]
        m = len(s)

        if m == 0:
            continue

        for j, tset in enumerate(target_sets):
            n = len(tset)
            if n == 0:
                continue

            c = len(s.intersection(tset))
            S[i, j] = (2.0 * c) / (m + n)

    return S


source_ids = np.arange(N1, dtype=np.int64)
target_ids = np.arange(N1, N1 + N2, dtype=np.int64)

# This can be a large matrix. For the user's DBP15K-sized data it is
# normally manageable in Colab, but monitor RAM.
print("Temporal similarity matrix shape:", (N1, N2))

time_sim = temporal_similarity_matrix(
    source_ids,
    target_ids,
    time_dict
)

print("Temporal similarity min/max:",
      float(time_sim.min()), float(time_sim.max()))
print("Fraction of exact temporal matches:",
      float(np.mean(time_sim == 1.0)))


Temporal similarity matrix shape: (19388, 19572)
Temporal similarity min/max: 0.0 1.0
Fraction of exact temporal matches: 0.00020878246929093833


In [12]:
# ============================================================
# 12. CSLS
# ============================================================

def csls_similarity(similarity, k=10):
    """CSLS for a source-to-target similarity matrix."""
    k = min(k, similarity.shape[0], similarity.shape[1])

    src_r = np.mean(
        np.partition(similarity, -k, axis=1)[:, -k:],
        axis=1
    )

    tgt_r = np.mean(
        np.partition(similarity, -k, axis=0)[-k:, :],
        axis=0
    )

    return 2.0 * similarity - src_r[:, None] - tgt_r[None, :]


def evaluate_embeddings(
    emb,
    eval_pairs,
    alpha=0.3,
    use_csls=True,
    csls_k=10
):
    source = emb[:N1]
    target = emb[N1:]

    # STEA uses inner-product embedding similarity.
    source_n = F.normalize(source, p=2, dim=1)
    target_n = F.normalize(target, p=2, dim=1)

    emb_sim = (
        source_n @ target_n.T
    ).detach().cpu().numpy()

    if use_csls:
        emb_sim = csls_similarity(emb_sim, csls_k)

    final_sim = (
        (1.0 - alpha) * emb_sim
        + alpha * time_sim
    )

    # Ground truth mapping from the evaluation set.
    gt = {}
    for a, b in eval_pairs:
        gt[int(a)] = int(b) - N1

    ranks_l2r = []
    for src, tgt in gt.items():
        scores = final_sim[src]
        order = np.argsort(-scores)
        rank = int(np.where(order == tgt)[0][0]) + 1
        ranks_l2r.append(rank)

    # Right-to-left
    gt_r = {int(b) - N1: int(a) for a, b in eval_pairs}
    ranks_r2l = []
    for tgt, src in gt_r.items():
        scores = final_sim[:, tgt]
        order = np.argsort(-scores)
        rank = int(np.where(order == src)[0][0]) + 1
        ranks_r2l.append(rank)

    def metrics(ranks):
        ranks = np.asarray(ranks)
        return {
            "Hits@1": float(np.mean(ranks <= 1)),
            "Hits@5": float(np.mean(ranks <= 5)),
            "Hits@10": float(np.mean(ranks <= 10)),
            "MRR": float(np.mean(1.0 / ranks)),
        }

    return {
        "left_to_right": metrics(ranks_l2r),
        "right_to_left": metrics(ranks_r2l),
        "similarity": final_sim,
    }


def print_metrics(result, title="Evaluation"):
    print("\n" + title)
    print("-" * len(title))

    for direction in ["left_to_right", "right_to_left"]:
        print(direction)
        for k, v in result[direction].items():
            print(f"  {k:8s}: {v*100:.2f}%" if k.startswith("Hits")
                  else f"  {k:8s}: {v:.4f}")


In [13]:
# ============================================================
# 13. Mutual nearest-neighbor pseudo-seed generation
# ============================================================

def mutual_nearest_pairs(similarity, existing_pairs=None):
    """
    STEA's iterative strategy:
    a pair is added when the two entities are mutually nearest neighbors.

    Existing pairs are excluded.
    """
    src_best = np.argmax(similarity, axis=1)
    tgt_best = np.argmax(similarity, axis=0)

    existing = set()
    if existing_pairs is not None:
        existing = {(int(a), int(b)) for a, b in existing_pairs}

    new_pairs = []

    for i, j in enumerate(src_best):
        if tgt_best[j] == i:
            pair = (i, N1 + int(j))
            if pair not in existing:
                new_pairs.append(pair)

    return np.asarray(new_pairs, dtype=np.int64).reshape(-1, 2)


def get_combined_similarity(emb, alpha=ALPHA, use_csls=True):
    source = F.normalize(emb[:N1], p=2, dim=1)
    target = F.normalize(emb[N1:], p=2, dim=1)

    emb_sim = (source @ target.T).detach().cpu().numpy()

    if use_csls:
        emb_sim = csls_similarity(emb_sim, CSLS_K)

    return (1.0 - alpha) * emb_sim + alpha * time_sim


In [14]:
# ============================================================
# 14. Train STEA
# ============================================================

def train_one_iteration(model, optimizer, train_pairs, epochs=1200):
    model.train()

    train_pairs_np = np.asarray(train_pairs, dtype=np.int64)

    if len(train_pairs_np) == 0:
        raise ValueError("No training pairs.")

    history = []

    # STEA uses a relatively small seed set in the paper (200–5000).
    # Computing the loss over all current seeds in one forward pass avoids
    # retaining multiple autograd graphs.
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        emb = model(adj)

        # Shuffle the seed order before negative sampling.
        perm = np.random.permutation(len(train_pairs_np))
        shuffled_pairs = train_pairs_np[perm]

        loss = stea_triplet_loss(
            emb,
            shuffled_pairs,
            margin=MARGIN
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        mean_loss = float(loss.detach().cpu())
        history.append(mean_loss)

        if epoch == 1 or epoch % 100 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:4d}/{epochs} | "
                f"loss={mean_loss:.6f}"
            )

    return history


# Start with the supervised seed pairs.
current_train_pairs = seed_pairs.copy()

all_histories = []

n_iterations = ITERATIONS if USE_ITERATIVE_STEA else 1

for iteration in range(1, n_iterations + 1):
    print("\n" + "=" * 70)
    print(f"STEA iteration {iteration}/{n_iterations}")
    print("Training pairs:", len(current_train_pairs))
    print("=" * 70)

    history = train_one_iteration(
        model,
        optimizer,
        current_train_pairs,
        epochs=EPOCHS_PER_ITER
    )
    all_histories.append(history)

    model.eval()
    with torch.no_grad():
        final_emb = model(adj)

    # Evaluate after each iteration.
    result = evaluate_embeddings(
        final_emb,
        eval_pairs,
        alpha=ALPHA,
        use_csls=True,
        csls_k=CSLS_K
    )
    print_metrics(result, f"Evaluation after iteration {iteration}")

    if not USE_ITERATIVE_STEA:
        break

    # Generate new mutual nearest-neighbor pairs.
    combined_sim = result["similarity"]
    new_pairs = mutual_nearest_pairs(
        combined_sim,
        existing_pairs=current_train_pairs
    )

    print("New mutual-NN pairs:", len(new_pairs))

    if len(new_pairs) == 0:
        print("No new pairs found; stopping iterative learning.")
        break

    current_train_pairs = np.vstack([
        current_train_pairs,
        new_pairs
    ])

print("\nFinal training-pair count:", len(current_train_pairs))



STEA iteration 1/5
Training pairs: 200
Epoch    1/1200 | loss=2.810583
Epoch  100/1200 | loss=0.006573
Epoch  200/1200 | loss=0.001151
Epoch  300/1200 | loss=0.000000
Epoch  400/1200 | loss=0.002463
Epoch  500/1200 | loss=0.000536
Epoch  600/1200 | loss=0.000000
Epoch  700/1200 | loss=0.000533
Epoch  800/1200 | loss=0.000000
Epoch  900/1200 | loss=0.000000
Epoch 1000/1200 | loss=0.002245
Epoch 1100/1200 | loss=0.000000
Epoch 1200/1200 | loss=0.000000

Evaluation after iteration 1
----------------------------
left_to_right
  Hits@1  : 4.00%
  Hits@5  : 11.69%
  Hits@10 : 16.65%
  MRR     : 0.0839
right_to_left
  Hits@1  : 4.20%
  Hits@5  : 11.46%
  Hits@10 : 16.66%
  MRR     : 0.0852
New mutual-NN pairs: 4354

STEA iteration 2/5
Training pairs: 4554
Epoch    1/1200 | loss=0.616483
Epoch  100/1200 | loss=0.003175
Epoch  200/1200 | loss=0.000907
Epoch  300/1200 | loss=0.002875
Epoch  400/1200 | loss=0.002293
Epoch  500/1200 | loss=0.002881
Epoch  600/1200 | loss=0.001147
Epoch  700/1200 

In [15]:
# ============================================================
# 15. Final evaluation
# ============================================================

model.eval()

with torch.no_grad():
    final_embeddings = model(adj)

final_result = evaluate_embeddings(
    final_embeddings,
    eval_pairs,
    alpha=ALPHA,
    use_csls=True,
    csls_k=CSLS_K
)

print_metrics(final_result, "FINAL STEA RESULT")

print("\nConfiguration")
print("-------------")
print("TRAIN_SEED:", TRAIN_SEED)
print("DIM:", DIM)
print("DEPTH:", DEPTH)
print("MARGIN:", MARGIN)
print("DROPOUT:", DROPOUT)
print("ALPHA:", ALPHA)
print("LR:", LR)
print("EPOCHS_PER_ITER:", EPOCHS_PER_ITER)
print("ITERATIONS:", ITERATIONS if USE_ITERATIVE_STEA else 1)
print("CSLS_K:", CSLS_K)



FINAL STEA RESULT
-----------------
left_to_right
  Hits@1  : 6.10%
  Hits@5  : 16.76%
  Hits@10 : 23.87%
  MRR     : 0.1203
right_to_left
  Hits@1  : 6.07%
  Hits@5  : 16.35%
  Hits@10 : 23.23%
  MRR     : 0.1185

Configuration
-------------
TRAIN_SEED: 200
DIM: 100
DEPTH: 2
MARGIN: 3.0
DROPOUT: 0.3
ALPHA: 0.3
LR: 0.005
EPOCHS_PER_ITER: 1200
ITERATIONS: 5
CSLS_K: 10


In [16]:
# ============================================================
# 16. Save checkpoint
# ============================================================

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/stea_zh_en_checkpoint.pt"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "TRAIN_SEED": TRAIN_SEED,
        "DIM": DIM,
        "DEPTH": DEPTH,
        "MARGIN": MARGIN,
        "DROPOUT": DROPOUT,
        "ALPHA": ALPHA,
        "LR": LR,
        "EPOCHS_PER_ITER": EPOCHS_PER_ITER,
        "ITERATIONS": ITERATIONS,
        "CSLS_K": CSLS_K,
    },
    "current_train_pairs": current_train_pairs,
    "final_metrics": {
        "left_to_right": final_result["left_to_right"],
        "right_to_left": final_result["right_to_left"],
    }
}, CHECKPOINT_PATH)

print("Saved:", CHECKPOINT_PATH)


Saved: /content/drive/MyDrive/stea_zh_en_checkpoint.pt


## Notes for interpreting the result

For your experiment, record at least:

- Hits@1
- Hits@5
- Hits@10
- MRR
- left-to-right and right-to-left results
- number of initial alignment seeds
- number of pseudo-seeds added by STEA
- `alpha`
- whether iterative learning was enabled

### Important comparison issue

STEA was originally evaluated on **English temporal KGs** such as DICEWS and YAGO-WIKI50K. Your `fr_en` dataset is a **cross-lingual temporal KG** dataset. Therefore, the original paper's reported scores should not be treated as directly comparable to your `fr_en` scores.

For your thesis experiment, the useful comparison is to run **RREA, TEA-GNN, TREA, and STEA under the same `fr_en` data split and evaluation protocol**, then compare their behavior under the same cross-lingual setting.

Also keep the implementation/protocol consistent across methods. In particular, document exactly which pairs are used as supervised seeds and which pairs are used for evaluation.
